# Example Data

Two ways to get example data:

1. **Bundled datasets** — the same collection SasView ships, curated with the model that fits each one and sensible starting parameters.
2. **Simulated data** — computed from any sasmodels model, with the ground truth attached, so you can check that a fit recovers it.

The bundled files have not been added into this repository.
They are located inside the installed `sasdata` package, which is a dependency.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sans_fitter import SANSFitter, data_ops, examples

## What is available

In [ ]:
examples.describe()

In [ ]:
# Full detail for one entry, including facts read live from the file.
examples.describe('silica_spheres')

In [ ]:
# Filter by tag to find a dataset of a particular kind.
for tag in ('measured', 'simulated', 'structure-factor', 'resolution', 'polydispersity'):
    print(f'{tag:18s} {examples.list_examples(tag=tag)}')

## The gallery

Every bundled dataset, plotted on log axes with its error bars.

In [ ]:
names = examples.list_examples()
n_cols = 3
n_rows = -(-len(names) // n_cols)  # ceiling division

fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=names,
                    horizontal_spacing=0.07, vertical_spacing=0.09)

for i, name in enumerate(names):
    data = examples.load(name)
    row, col = i // n_cols + 1, i % n_cols + 1
    error_y = None
    if data.dy is not None and np.any(np.nan_to_num(data.dy) != 0):
        error_y = dict(type='data', array=data.dy, visible=True, thickness=0.6, width=0)
    fig.add_trace(
        go.Scatter(x=data.x, y=data.y, error_y=error_y, mode='markers',
                   marker=dict(size=3), name=name, showlegend=False),
        row=row, col=col,
    )
    fig.update_xaxes(type='log', title_text='Q (Å⁻¹)', row=row, col=col)
    fig.update_yaxes(type='log', title_text='I(Q)', row=row, col=col)

fig.update_layout(height=300 * n_rows, width=1050,
                  title='Bundled example datasets', font=dict(size=10))
fig.update_annotations(font_size=12)
fig

## One-line fit from a preset

`load_fitter()` returns a `SANSFitter` with the data, model, structure factor, polydispersity and starting parameters already set.

In [ ]:
fitter = examples.load_fitter('silica_spheres')
result = fitter.fit()
fitter.get_params()

In [ ]:
fitter.plot_results(show_residuals=True)

The starting values in a preset are coarse — they put the optimiser in the right basin, they are not published results. For measured data there is no ground truth to compare against, and `get_example(name).truth` is `None` to say so.

In [ ]:
print('silica_spheres truth:', examples.get_example('silica_spheres').truth)
print('cylinder truth:      ', examples.get_example('cylinder').truth)

## What resolution smearing does

`sphere` and `sphere_smeared` are the same sample; only the second carries a `dQ` column. Smearing washes out the sharp form-factor minima.

In [ ]:
plain = examples.load('sphere')
smeared = examples.load('sphere_smeared')

fig = go.Figure()
fig.add_trace(go.Scatter(x=plain.x, y=plain.y, mode='markers',
                         marker=dict(size=4), name='no dQ'))
fig.add_trace(go.Scatter(x=smeared.x, y=smeared.y, mode='markers',
                         marker=dict(size=4), name='pinhole dQ'))
fig.update_layout(xaxis_type='log', yaxis_type='log', width=800, height=450,
                  xaxis_title='Q (Å⁻¹)', yaxis_title='I(Q)',
                  title='Same spheres, with and without resolution')
fig

## Simulated data with known truth

`simulate()` works for any sasmodels model and attaches the generating parameters as `data.truth`. That makes it a self-checking exercise: state the answer, then confirm the fit finds it.

In [ ]:
truth = dict(radius=50.0, sld=4.0, sld_solvent=1.0, scale=0.1, background=0.001)
data = examples.simulate('sphere', npoints=80, noise=0.02, seed=3, **truth)

print('generated with radius =', data.truth['radius'], 'Å')

fitter = SANSFitter()
fitter.set_data(data)
fitter.set_model('sphere')
fitter.set_param('radius', value=30, min=5, max=300, vary=True)
fitter.set_param('scale', value=0.05, min=1e-4, max=10, vary=True)
fitter.set_param('background', value=0.001, min=0, max=1, vary=True)
fitter.set_param('sld', value=4.0, vary=False)
fitter.set_param('sld_solvent', value=1.0, vary=False)

result = fitter.fit()
recovered = result['parameters']['radius']['value']
print(f"\nstarted at 30 Å, recovered {recovered:.2f} Å, truth was {data.truth['radius']} Å")
print(f"reduced chi-squared = {result['chisq']:.2f}  (near 1 means the error bars are honest)")

In [ ]:
fitter.plot_results(show_residuals=True)

### Controlling the simulation

Noise, Q range, resolution and polydispersity are all independent variables.

In [ ]:
variants = {
    'clean': dict(noise=0.005),
    'noisy': dict(noise=0.10),
    'smeared (dq=0.15)': dict(noise=0.005, dq=0.15),
    'polydisperse (15%)': dict(noise=0.005, radius_pd=0.15),
}

fig = go.Figure()
for label, kwargs in variants.items():
    d = examples.simulate('sphere', radius=50, scale=0.1, npoints=120, seed=1, **kwargs)
    fig.add_trace(go.Scatter(x=d.x, y=d.y, mode='markers', marker=dict(size=3), name=label))

fig.update_layout(xaxis_type='log', yaxis_type='log', width=850, height=500,
                  xaxis_title='Q (Å⁻¹)', yaxis_title='I(Q)',
                  title='One 50 Å sphere, four ways')
fig

Note that `radius_pd=0.15` is enough on its own — `simulate()` fills in the `_pd_n` / `_pd_type` companions, which sasmodels needs before it will integrate over the distribution at all.

## A sample and background pair

`simulate_pair()` produces a matched sample and background on an identical Q grid — which is what `data_ops` requires — so background subtraction can be demonstrated end to end.

In [ ]:
sample, background = examples.simulate_pair(
    'sphere', radius=50, sld=4.0, sld_solvent=1.0, scale=0.1,
    background_level=0.5, npoints=80, seed=2,
)
subtracted = data_ops.subtract(sample, background)

fig = go.Figure()
for label, d in (('sample', sample), ('background', background), ('subtracted', subtracted)):
    fig.add_trace(go.Scatter(x=d.x, y=d.y, mode='markers', marker=dict(size=4), name=label))
fig.update_layout(xaxis_type='log', yaxis_type='log', width=850, height=500,
                  xaxis_title='Q (Å⁻¹)', yaxis_title='I(Q)',
                  title='Background subtraction')
fig

In [ ]:
fitter = SANSFitter()
fitter.set_data(subtracted)
fitter.set_model('sphere')
fitter.set_param('radius', value=30, min=5, max=300, vary=True)
fitter.set_param('scale', value=0.05, min=1e-4, max=10, vary=True)
fitter.set_param('background', value=0.0, min=-1, max=1, vary=True)
fitter.set_param('sld', value=4.0, vary=False)
fitter.set_param('sld_solvent', value=1.0, vary=False)

result = fitter.fit()
print(f"radius after subtraction: {result['parameters']['radius']['value']:.2f} Å (truth 50 Å)")

## Where the files live

If you want to hand a bundled file to something else, ask for its path.

In [ ]:
print(examples.example_path('silica_spheres'))

# Equivalent to examples.load('silica_spheres'):
same = data_ops.load(examples.example_path('silica_spheres'))
print('points:', len(same.x))